In [1]:
import json
import os
from typing import List, Dict
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW, get_linear_schedule_with_warmup

# Constants and Configurations
MODEL_NAME = 't5-base'  # You can choose 't5-base' or 't5-large' based on your resources/
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 5e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
def load_json(file_path: str) -> Dict[str, List[str]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


# Paths to your JSON files
original_data_path = 'input.json'  # Replace with your path
generated_data_path = 'output.json'  # Replace with your path

# Load the data
original_data = load_json(original_data_path)
generated_data = load_json(generated_data_path)

print(f"Loaded {len(original_data)} entries from original_data.json")
print(f"Loaded {len(generated_data)} entries from generated_descriptions.json")


Loaded 51 entries from original_data.json
Loaded 51 entries from generated_descriptions.json


In [3]:
def prepare_data(original_data: Dict[str, List[str]], generated_data: Dict[str, str]) -> (List[str], List[str]):
    inputs = []
    targets = []
    
    for image_key in original_data:
        q_a_list = original_data[image_key]
        description = generated_data.get(image_key, None)
        if description:
            # Concatenate Q&A responses into a single string
            concatenated_q_a = ' '.join(q_a_list)
            inputs.append(concatenated_q_a)
            targets.append(description)
        else:
            print(f"Warning: No generated description for image {image_key}")
    
    print(f"Prepared {len(inputs)} input-target pairs for training.")
    return inputs, targets

inputs, targets = prepare_data(original_data, generated_data)


Prepared 51 input-target pairs for training.


In [4]:
class AdDataset(Dataset):
    def __init__(self, inputs: List[str], targets: List[str], tokenizer: T5Tokenizer, 
                 max_input_length: int = 512, max_target_length: int = 256):
        self.inputs = inputs
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_text = self.inputs[idx]
        target_text = self.targets[idx]

        input_encoding = self.tokenizer.encode_plus(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target_encoding = self.tokenizer.encode_plus(
            target_text,
            max_length=self.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        labels = target_encoding['input_ids']
        labels[labels == self.tokenizer.pad_token_id] = -100  # Replace pad token id with -100

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels.squeeze()
        }


In [7]:
# Initialize tokenizer and model
# tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
# model = model.to(DEVICE)

# print(f"Model {MODEL_NAME} loaded and moved to {DEVICE}")


In [8]:
save_path = 'fine_tuned_t5_model'
# Load the fine-tuned model and tokenizer
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(save_path)
model = fine_tuned_model.to(DEVICE)
tokenizer = T5Tokenizer.from_pretrained(save_path)
model = model.to(DEVICE)
print("Fine-tuned model and tokenizer loaded.")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Fine-tuned model and tokenizer loaded.


In [9]:
# Create the dataset
dataset = AdDataset(inputs, targets, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)

# Create DataLoader
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"DataLoader created with batch size {BATCH_SIZE}")


DataLoader created with batch size 4


In [10]:
# Initialize optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Total training steps
total_steps = len(train_loader) * EPOCHS

# Scheduler for learning rate decay
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=0, 
                                            num_training_steps=total_steps)

print("Optimizer and scheduler set up.")


c:\Users\Codew\anaconda3\envs\nlp\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Optimizer and scheduler set up.


In [12]:
def train_epoch(model, tokenizer, data_loader, optimizer, scheduler, device, epoch):
    model.train()
    loop = tqdm(data_loader, leave=True, desc=f'Epoch {epoch}/{EPOCHS}')
    total_loss = 0
    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss
        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch} completed. Average Loss: {avg_loss:.4f}")

# Training across multiple epochs
for epoch in range(1, EPOCHS + 3):
    train_epoch(model, tokenizer, train_loader, optimizer, scheduler, DEVICE, epoch)


Epoch 1/3:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch 1 completed. Average Loss: 1.7623


Epoch 2/3:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch 2 completed. Average Loss: 1.7566


Epoch 3/3:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch 3 completed. Average Loss: 1.7532


Epoch 4/3:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch 4 completed. Average Loss: 1.7492


Epoch 5/3:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch 5 completed. Average Loss: 1.7446


In [ ]:
save_path = 'fine_tuned_t5_model'
# Load the fine-tuned model and tokenizer
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(save_path)
model = fine_tuned_model.to(DEVICE)
tokenizer = T5Tokenizer.from_pretrained(save_path)
model = model.to(DEVICE)
print("Fine-tuned model and tokenizer loaded.")



Model and tokenizer saved to fine_tuned_t5_model


In [ ]:
import json
import os
from typing import List, Dict1
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW, get_linear_schedule_with_warmup


In [10]:
save_path = 'fine_tuned_t5_model'
# Load the fine-tuned model and tokenizer
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(save_path)
fine_tuned_model = fine_tuned_model.to(DEVICE)
fine_tuned_tokenizer = T5Tokenizer.from_pretrained(save_path)

print("Fine-tuned model and tokenizer loaded.")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Fine-tuned model and tokenizer loaded.


In [13]:
def load_json(file_path: str) -> Dict[str, List[str]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


In [14]:
def generate_description(model, tokenizer, q_a_list: List[str], device, max_input_length=512, max_target_length=256) -> str:
    model.eval()
    with torch.no_grad():
        # Concatenate Q&A responses
        input_text = ' '.join(q_a_list)
        input_encoding = tokenizer.encode_plus(
            input_text,
            max_length=max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = input_encoding['input_ids'].to(device)
        attention_mask = input_encoding['attention_mask'].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_target_length,
            num_beams=4,
            early_stopping=True
        )

        description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return description


In [15]:
new_input_path = r"C:\Users\Codew\Downloads\annotations_images\image\1\1.json" # Replace with your path

# Load the new input data
new_input_data = load_json(new_input_path)

print(f"Loaded {len(new_input_data)} entries from {new_input_path}")


Loaded 1889 entries from C:\Users\Codew\Downloads\annotations_images\image\1\1.json


In [10]:
# Dictionary to store the generated descriptions
output_data = {}

for image_key in tqdm(new_input_data, desc="Generating descriptions"):
    q_a_list = new_input_data[image_key]
    description = generate_description(fine_tuned_model, fine_tuned_tokenizer, q_a_list, DEVICE)
    output_data[image_key] = description

print("Description generation completed.")


Generating descriptions:   0%|          | 0/7710 [00:00<?, ?it/s]

Description generation completed.


In [11]:
# Path to save the output JSON file
output_json_path = 'output_descriptions.json'  # Replace with your desired path

# Save the output data to JSON
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Generated descriptions saved to {output_json_path}")


Generated descriptions saved to output_descriptions.json



1. **Data Preparation and Loading**: Load the original Q&A data and the generated descriptions.
2. **Dataset Preparation**: Create a custom dataset compatible with the model.
3. **Model Fine-Tuning**: Fine-tune a pre-trained T5 model using Hugging Face's Transformers library.
4. **Inference**: Generate descriptions for new Q&A data using the fine-tuned model.
5. **Saving Results**: Save the generated descriptions to a new JSON file.

### **Prerequisites**

Before you begin, ensure you have the following libraries installed. You can install them directly from your Jupyter Notebook using `pip`:

```python
!pip install transformers torch tqdm
```

### **Overview**

We'll use the `transformers` library by Hugging Face, specifically the T5 model, which is well-suited for text generation tasks. The process involves fine-tuning the model on your dataset and then using it to generate descriptions for new data.

---

### **1. Imports and Setup**

First, import all necessary libraries and set up configurations such as device selection (CPU or GPU).

```python
import json
import os
from typing import List, Dict
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import T5Tokenizer, T5ForConditionalGeneration, AdamW, get_linear_schedule_with_warmup

# Constants and Configurations
MODEL_NAME = 't5-base'  # You can choose 't5-base' or 't5-large' based on your resources
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 5e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
```

---

### **2. Data Loading and Preparation**

Load your original Q&A data and the generated descriptions from JSON files. Ensure your JSON files are structured correctly based on your initial data.

Assuming you have two JSON files:

- `original_data.json`: Contains image keys mapped to lists of Q&A responses.
- `generated_descriptions.json`: Contains image keys mapped to their corresponding generated descriptions.

```python
def load_json(file_path: str) -> Dict[str, List[str]]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# Paths to your JSON files
original_data_path = 'original_data.json'  # Replace with your path
generated_data_path = 'generated_descriptions.json'  # Replace with your path

# Load the data
original_data = load_json(original_data_path)
generated_data = load_json(generated_data_path)

print(f"Loaded {len(original_data)} entries from original_data.json")
print(f"Loaded {len(generated_data)} entries from generated_descriptions.json")
```

> **Note:** Replace `'original_data.json'` and `'generated_descriptions.json'` with the actual paths to your JSON files.

---

### **3. Prepare Input and Target Lists**

Create lists of inputs (concatenated Q&A responses) and targets (generated descriptions) for training.

```python
def prepare_data(original_data: Dict[str, List[str]], generated_data: Dict[str, str]) -> (List[str], List[str]):
    inputs = []
    targets = []
    
    for image_key in original_data:
        q_a_list = original_data[image_key]
        description = generated_data.get(image_key, None)
        if description:
            # Concatenate Q&A responses into a single string
            concatenated_q_a = ' '.join(q_a_list)
            inputs.append(concatenated_q_a)
            targets.append(description)
        else:
            print(f"Warning: No generated description for image {image_key}")
    
    print(f"Prepared {len(inputs)} input-target pairs for training.")
    return inputs, targets

inputs, targets = prepare_data(original_data, generated_data)
```

---

### **4. Define a Custom Dataset**

Create a custom `Dataset` class that tokenizes the inputs and targets.

```python
class AdDataset(Dataset):
    def __init__(self, inputs: List[str], targets: List[str], tokenizer: T5Tokenizer, 
                 max_input_length: int = 512, max_target_length: int = 256):
        self.inputs = inputs
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_text = self.inputs[idx]
        target_text = self.targets[idx]

        input_encoding = self.tokenizer.encode_plus(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target_encoding = self.tokenizer.encode_plus(
            target_text,
            max_length=self.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        labels = target_encoding['input_ids']
        labels[labels == self.tokenizer.pad_token_id] = -100  # Replace pad token id with -100

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels.squeeze()
        }
```

---

### **5. Initialize Tokenizer and Model**

Load the pre-trained T5 tokenizer and model and move the model to the selected device.

```python
# Initialize tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

print(f"Model {MODEL_NAME} loaded and moved to {DEVICE}")
```

---

### **6. Create DataLoader**

Instantiate the custom dataset and create a `DataLoader` for batching.

```python
# Create the dataset
dataset = AdDataset(inputs, targets, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)

# Create DataLoader
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"DataLoader created with batch size {BATCH_SIZE}")
```

---

### **7. Set Up Optimizer and Scheduler**

Configure the optimizer and learning rate scheduler for training.

```python
# Initialize optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Total training steps
total_steps = len(train_loader) * EPOCHS

# Scheduler for learning rate decay
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=0, 
                                            num_training_steps=total_steps)

print("Optimizer and scheduler set up.")
```

---

### **8. Training Loop**

Define the training loop to fine-tune the model on your dataset.

```python
def train_epoch(model, tokenizer, data_loader, optimizer, scheduler, device, epoch):
    model.train()
    loop = tqdm(data_loader, leave=True, desc=f'Epoch {epoch}/{EPOCHS}')
    total_loss = 0
    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch} completed. Average Loss: {avg_loss:.4f}")

# Training across multiple epochs
for epoch in range(1, EPOCHS + 1):
    train_epoch(model, tokenizer, train_loader, optimizer, scheduler, DEVICE, epoch)
```

> **Note:** Training transformer models can be computationally intensive. Ensure you have access to a GPU for faster training. If you're using a CPU, consider reducing the `BATCH_SIZE` or the number of `EPOCHS` to accommodate longer training times.

---

### **9. Saving the Fine-Tuned Model**

After training, save the fine-tuned model and tokenizer for future use.

```python
# Define path to save the fine-tuned model
save_path = 'fine_tuned_t5_model'  # You can change this path as needed

# Create the directory if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Save the model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to {save_path}")
```

---

### **10. Inference: Generating Descriptions for New Data**

Now that the model is fine-tuned, use it to generate descriptions for new Q&A data.

#### **a. Load the Fine-Tuned Model and Tokenizer**

```python
# Load the fine-tuned model and tokenizer
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(save_path)
fine_tuned_model = fine_tuned_model.to(DEVICE)
fine_tuned_tokenizer = T5Tokenizer.from_pretrained(save_path)

print("Fine-tuned model and tokenizer loaded.")
```

#### **b. Define the Inference Function**

Create a function to generate descriptions given a list of Q&A responses.

```python
def generate_description(model, tokenizer, q_a_list: List[str], device, max_input_length=512, max_target_length=256) -> str:
    model.eval()
    with torch.no_grad():
        # Concatenate Q&A responses
        input_text = ' '.join(q_a_list)
        input_encoding = tokenizer.encode_plus(
            input_text,
            max_length=max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = input_encoding['input_ids'].to(device)
        attention_mask = input_encoding['attention_mask'].to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_target_length,
            num_beams=4,
            early_stopping=True
        )

        description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return description
```

#### **c. Load New Q&A Data for Inference**

Assume you have a new JSON file `new_input.json` structured similarly to `original_data.json`.

```python
# Path to the new input JSON file
new_input_path = 'new_input.json'  # Replace with your path

# Load the new input data
new_input_data = load_json(new_input_path)

print(f"Loaded {len(new_input_data)} entries from {new_input_path}")
```

#### **d. Generate Descriptions for New Data**

Iterate over the new data and generate descriptions.

```python
# Dictionary to store the generated descriptions
output_data = {}

for image_key in tqdm(new_input_data, desc="Generating descriptions"):
    q_a_list = new_input_data[image_key]
    description = generate_description(fine_tuned_model, fine_tuned_tokenizer, q_a_list, DEVICE)
    output_data[image_key] = description

print("Description generation completed.")
```

#### **e. Save the Generated Descriptions to a New JSON File**

```python
# Path to save the output JSON file
output_json_path = 'output_descriptions.json'  # Replace with your desired path

# Save the output data to JSON
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Generated descriptions saved to {output_json_path}")
```

---

### **11. Full Workflow Execution**

For better organization, here's how you can sequentially execute all the steps in your Jupyter Notebook. Ensure you run each cell in order.

1. **Install Dependencies**

    ```python
    !pip install transformers torch tqdm
    ```

2. **Imports and Setup**

    *(Run the code from Section 1 above.)*

3. **Data Loading and Preparation**

    *(Run the code from Section 2 and 3 above.)*

4. **Define Custom Dataset**

    *(Run the code from Section 4 above.)*

5. **Initialize Tokenizer and Model**

    *(Run the code from Section 5 above.)*

6. **Create DataLoader**

    *(Run the code from Section 6 above.)*

7. **Set Up Optimizer and Scheduler**

    *(Run the code from Section 7 above.)*

8. **Training Loop**

    *(Run the code from Section 8 above.)*

9. **Save the Fine-Tuned Model**

    *(Run the code from Section 9 above.)*

10. **Inference: Generate Descriptions for New Data**

    *(Run the code from Section 10.a to 10.e above.)*

---

### **Additional Considerations**

- **Model Choice**: While `t5-small` is efficient and faster to train, you can opt for larger models like `t5-base` or `t5-large` for potentially better performance, provided you have the necessary computational resources.

- **Hyperparameter Tuning**: Adjust `BATCH_SIZE`, `EPOCHS`, and `LEARNING_RATE` based on your dataset size and available hardware to optimize training.

- **Handling Large Datasets**: If your dataset is large, consider using techniques like gradient accumulation to effectively increase the batch size without exceeding memory limits.

- **Evaluation**: For a more robust training process, consider splitting your data into training and validation sets to monitor the model's performance and prevent overfitting.

- **Saving Intermediate Models**: During training, periodically save model checkpoints to prevent data loss and enable resuming training if interrupted.

---

### **Conclusion**

This comprehensive guide provides you with all the necessary steps to fine-tune a pre-trained T5 model on your dataset and generate descriptive summaries based on Q&A data. By following these steps in your Jupyter Notebook, you should be able to train the model and perform inference seamlessly.
